# 05 — Combined Clinical + Per-Image PCA Clustering

Clusters OASIS-1 patients by combining the clinical variables with the
**476 per-image PCA components** (from notebook 03, 59.2 % variance threshold
applied independently to each of the 5 image types).

A Generalized Gower distance matrix handles the mixed variable types
across all retained columns together.

**Inputs**
- `oasis_combined.csv` — clinical columns + CDR
- `pca_per_image_features.csv` — 476 per-image PCA features (patient_id + cor/sag/tra components)

**Output:** `combined_clinical_pca_per_image_results.csv` — `patient_id`, `cluster`, `CDR`

In [1]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "robust-mixed-dist", "kmedoids", "scikit-learn", "-q"],
    check=True
)

CompletedProcess(args=['C:\\Users\\aregk\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', 'robust-mixed-dist', 'kmedoids', 'scikit-learn', '-q'], returncode=0)

In [2]:
import os
import numpy as np
import pandas as pd

from robust_mixed_dist.mixed import generalized_gower_dist_matrix
import kmedoids

## Step 1 — Load and merge data

Load both source files and merge on `patient_id`.  CDR is split off
immediately and never used during clustering.

In [3]:
DATA_DIR      = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
CLINICAL_CSV  = os.path.join(DATA_DIR, "method_1_HOG_PCA", "oasis_combined.csv")
PCA_IMG_CSV   = os.path.join(DATA_DIR, "notebooks", "pca_per_image_features.csv")
OUTPUT_CSV    = os.path.join(DATA_DIR, "notebooks", "combined_clinical_pca_per_image_results.csv")

CLINICAL_COLS = ["Age", "M/F", "Hand", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF", "Delay"]

# Load clinical data
raw_clin  = pd.read_csv(CLINICAL_CSV)
clin      = raw_clin[["patient_id"] + CLINICAL_COLS].copy()
cdr       = raw_clin[["patient_id", "CDR"]].copy()   # evaluation only

# Load per-image PCA features
pca_img   = pd.read_csv(PCA_IMG_CSV)
pca_cols  = [c for c in pca_img.columns if c != "patient_id"]

# Merge on patient_id
df = clin.merge(pca_img, on="patient_id", how="inner")

print(f"Clinical rows    : {len(clin)}")
print(f"Per-image PCA rows: {len(pca_img)}  ({len(pca_cols)} feature columns)")
print(f"Merged rows      : {len(df)}  ({df.shape[1]-1} feature columns)")
df.head(3)

Clinical rows    : 416
Per-image PCA rows: 416  (476 feature columns)
Merged rows      : 416  (486 feature columns)


,patient_id,Age,M/F,Hand,Educ,SES,MMSE,eTIV,nWBV,ASF,...,sbj_sag_33,sbj_sag_34,sbj_sag_35,sbj_sag_36,sbj_sag_37,sbj_sag_38,sbj_sag_39,sbj_sag_40,sbj_sag_41,sbj_sag_42
0,OAS1_0001,74,F,R,2.0,3.0,29.0,1344,0.743,1.306,...,0.550983,0.740857,-0.184640,0.892996,-0.273372,-0.359235,0.166910,0.345266,-0.403029,0.145446
1,OAS1_0002,55,F,R,4.0,1.0,29.0,1147,0.810,1.531,...,-0.116964,-0.825775,-0.648605,-0.112522,-0.033159,0.131882,0.325924,-0.309457,-0.112991,0.758400
2,OAS1_0003,73,F,R,4.0,3.0,27.0,1454,0.708,1.207,...,0.004772,-0.167228,-0.119309,0.239583,0.707012,-0.525547,-0.221539,0.171652,-0.372457,-1.030444


## Step 2 — Missing value analysis

The per-image PCA columns have no missing values.  Only the clinical columns
need attention:

| Column | Situation | Action |
|--------|-----------|--------|
| `Delay` | 100 % missing | **Drop** |
| `Hand` | Constant (`R`) | **Drop** — zero variance |
| `Educ`, `SES`, `MMSE` | ~43–48 % missing | **Impute** with median |
| All PCA columns | 0 % missing | Keep as-is |

In [4]:
missing    = df.isnull().sum().rename("missing")
pct        = (missing / len(df) * 100).round(1).rename("%")
unique_cnt = df.nunique().rename("unique_vals")

# Show clinical columns + any PCA column with missing values
show_cols = [c for c in df.columns
             if c in ["patient_id"] + CLINICAL_COLS or missing[c] > 0]
print(pd.concat([missing, pct, unique_cnt], axis=1).loc[show_cols].to_string())
print(f"\nPer-image PCA columns missing: {missing[pca_cols].sum()} total")

            missing      %  unique_vals
patient_id        0    0.0          416
Age               0    0.0           73
M/F               0    0.0            2
Hand              0    0.0            1
Educ            181   43.5            5
SES             200   48.1            5
MMSE            181   43.5           17
eTIV              0    0.0          301
nWBV              0    0.0          182
ASF               0    0.0          275
Delay           416  100.0            0

Per-image PCA columns missing: 0 total


In [5]:
df = df.drop(columns=["Delay", "Hand"])
print("Dropped: Delay (100% missing), Hand (constant 'R')")

for col in ["Educ", "SES", "MMSE"]:
    med = df[col].median()
    n_filled = df[col].isnull().sum()
    df[col] = df[col].fillna(med)
    print(f"  {col}: filled {n_filled} NaNs with median = {med}")

print(f"\nRemaining missing : {df.isnull().sum().sum()}")
print(f"Final feature set : {df.shape[1]-1} columns  "
      f"(7 clinical quantitative + 1 binary + {len(pca_cols)} per-image PCA)")

Dropped: Delay (100% missing), Hand (constant 'R')
  Educ: filled 181 NaNs with median = 3.0
  SES: filled 200 NaNs with median = 2.0
  MMSE: filled 181 NaNs with median = 29.0

Remaining missing : 0
Final feature set : 484 columns  (7 clinical quantitative + 1 binary + 476 per-image PCA)


## Step 3 — Generalized Gower distance matrix

Columns are ordered **quantitative → binary** as required by the function:

| Type | Columns | Count |
|------|---------|-------|
| Quantitative (`p1`) | Age, Educ, SES, MMSE, eTIV, nWBV, ASF + all per-image PCA | 483 |
| Binary (`p2`) | M/F (0=F, 1=M) | 1 |
| Multi-class (`p3`) | *(none)* | 0 |

In [6]:
df["MF_bin"] = df["M/F"].map({"F": 0, "M": 1}).astype(int)

quant_cols  = ["Age", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"] + pca_cols
binary_cols = ["MF_bin"]

p1 = len(quant_cols)    # 483
p2 = len(binary_cols)   # 1
p3 = 0

X = df[quant_cols + binary_cols].values.astype(float)

print(f"Feature matrix shape : {X.shape}")
print(f"  p1 (quantitative)  : {p1}  (7 clinical + {len(pca_cols)} per-image PCA)")
print(f"  p2 (binary)        : {p2}  (M/F)")
print(f"  p3 (multi-class)   : {p3}")

D = generalized_gower_dist_matrix(
    X,
    p1=p1, p2=p2, p3=p3,
    d1="minkowski",
    d2="sokal",
    d3="hamming",
    q=1
)

print(f"\nDistance matrix shape : {D.shape}")
print(f"Value range           : [{D.min():.4f}, {D.max():.4f}]")
print(f"Is symmetric          : {np.allclose(D, D.T)}")

Feature matrix shape : (416, 484)
  p1 (quantitative)  : 483  (7 clinical + 476 per-image PCA)
  p2 (binary)        : 1  (M/F)
  p3 (multi-class)   : 0

Distance matrix shape : (416, 416)
Value range           : [0.0000, 4.0785]
Is symmetric          : True


## Step 4 — K-medoids clustering (FasterPAM, k=4)

In [7]:
K = 4

result = kmedoids.fasterpam(D, medoids=K, random_state=42)
labels = np.array(result.labels)

print(f"Loss (sum of distances to medoids): {result.loss:.4f}")
print(f"Medoid patient IDs : {df['patient_id'].iloc[list(result.medoids)].tolist()}")
print()
sizes = pd.Series(labels).value_counts().sort_index().rename("count")
print("Cluster sizes:")
print(sizes.to_string())

Loss (sum of distances to medoids): 394.3828
Medoid patient IDs : ['OAS1_0356', 'OAS1_0289', 'OAS1_0207', 'OAS1_0127']

Cluster sizes:
0    117
1    139
2     63
3     97


## Step 5 — Evaluation: cluster vs CDR

In [8]:
results = df[["patient_id"]].copy()
results["cluster"] = labels
results = results.merge(cdr, on="patient_id", how="left")

print(f"Patients with CDR    : {results['CDR'].notnull().sum()}")
print(f"Patients without CDR : {results['CDR'].isnull().sum()}")
print()

has_cdr = results[results["CDR"].notnull()].copy()
has_cdr["CDR"] = has_cdr["CDR"].astype(str)

ct = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    margins=True,
    margins_name="Total"
)
print("Cluster × CDR (counts):")
ct

Patients with CDR    : 235
Patients without CDR : 181

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,33,17,11,0,61
1,64,22,8,1,95
2,17,12,4,0,33
3,21,19,5,1,46
Total,135,70,28,2,235


In [9]:
ct_norm = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    normalize="index"
).round(3)

print("Row-normalised (CDR proportion within each cluster):")
ct_norm

Row-normalised (CDR proportion within each cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,0.541,0.279,0.180,0.000
1,0.674,0.232,0.084,0.011
2,0.515,0.364,0.121,0.000
3,0.457,0.413,0.109,0.022


## Step 6 — Save results

In [10]:
results.to_csv(OUTPUT_CSV, index=False)

print(f"Saved : {OUTPUT_CSV}")
print(f"Shape : {results.shape}  (rows=patients, cols=patient_id, cluster, CDR)")
print()
print(results.head(5).to_string(index=False))

Saved : C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\combined_clinical_pca_per_image_results.csv
Shape : (416, 3)  (rows=patients, cols=patient_id, cluster, CDR)

patient_id  cluster  CDR
 OAS1_0001        1  0.0
 OAS1_0002        1  0.0
 OAS1_0003        0  0.5
 OAS1_0004        3  NaN
 OAS1_0005        2  NaN
